# Support client -- SOLUTIONS

Meme raisonnement que `interview/real_test/solutions/real_test_solutions.ipynb`,
applique a ce sujet. Relisez ce notebook de reference si un passage n'est
pas clair : le raisonnement est identique, seul le vocabulaire change.


---
## Question 1 -- Data analysis -- Solution


In [ ]:
import numpy as np
import pandas as pd

DATA = "../data/q1"

actors = pd.read_csv(f"{DATA}/agents.csv")
events = pd.concat(
    [pd.read_csv(f"{DATA}/tickets_{i}.csv") for i in range(1, 5)],
    ignore_index=True,
)

average_rating = actors["rating"].mean()
pct_second_language = (actors["second_language"] != "no").mean() * 100
success_rate = (events["status"] == "Success").mean() * 100

results = pd.DataFrame({
    "insight_type": [
        "average_agent_rating",
        "percentage_agents_with_second_language",
        "ticket_success_rate",
    ],
    "value": [average_rating, pct_second_language, success_rate],
})
results.to_csv("analysis_results.csv", index=False)
results


**RAISONNEMENT.** Trois insights independants -> trois calculs isoles,
assembles a la fin en `insight_type`/`value`. `condition.mean() * 100` pour
un pourcentage (moyenne d'un booleen = taux). Combiner les 4 fichiers
d'evenements AVANT de calculer le taux de succes (jamais moyenne de 4 taux
partiels si les fichiers n'ont pas le meme nombre de lignes).


---
## Question 2 -- Data collecting -- Solution


In [ ]:
DATA2 = "../data/q2"

actors2 = pd.read_csv(f"{DATA2}/agents.csv")
assets = pd.read_csv(f"{DATA2}/headsets.csv")
events2 = pd.concat(
    [pd.read_csv(f"{DATA2}/tickets_{i}.csv") for i in range(1, 5)],
    ignore_index=True,
)

TODAY = pd.Timestamp("2023-04-15")

upvote_cols = ['clarity_upvote_given', 'politeness_upvote_given', 'resolution_speed_upvote_given', 'follow_up_upvote_given']
events2["n_upvotes_this_event"] = events2[upvote_cols].sum(axis=1)
upvotes_per_actor = (
    events2.groupby("agent_id")["n_upvotes_this_event"].sum()
    .rename("number_of_upvotes")
)

assets["last_inspection_date"] = pd.to_datetime(assets["last_inspection_date"])
assets["days_since_inspection"] = (TODAY - assets["last_inspection_date"]).dt.days

collected = actors2.merge(
    assets[["headset_id", "model", "manufacture_year", "days_since_inspection"]],
    on="headset_id", how="left",
).rename(columns={"model": "headset_model", "manufacture_year": "headset_manufacture_year"})

collected["experience"] = 2023 - collected["started_year"]

collected = collected.merge(upvotes_per_actor, on="agent_id", how="left")
collected["number_of_upvotes"] = collected["number_of_upvotes"].fillna(0).astype(int)

collected = collected[[
    "agent_id", "headset_model", "headset_manufacture_year",
    "days_since_inspection", "age", "experience", "second_language", "rating",
    "net_worth_of_tips", "number_of_upvotes", "agent_class",
]]
collected.to_csv("collected.csv", index=False)
collected.head()


**RAISONNEMENT.** Repartir du schema de sortie demande, colonne par
colonne, et identifier sa source (directe / jointure / calcul / agregation).
`headset_model` et `headset_manufacture_year` viennent d'une
jointure sur `headset_id`. `experience` est une formule explicite de
l'enonce (`2023 - started_year`), a suivre a la lettre. `number_of_upvotes`
= somme des 4 colonnes booleennes par ticket, puis agregation par
agent -- `fillna(0)` pour les agents sans aucun ticket
(absents du merge -> NaN -> logiquement 0).


In [ ]:
import os
os.makedirs("../data/q2", exist_ok=True)
collected.to_csv("../data/q2/collected_reference.csv", index=False)


---
## Question 3 -- Data processing -- Solution


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

train_raw, test_raw = train_test_split(collected, test_size=0.30, random_state=42)
train_raw.to_csv("../data/train.csv", index=False)
test_raw.to_csv("../data/test.csv", index=False)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

mean_age = round(train["age"].mean())
train["age"] = train["age"].fillna(mean_age)
test["age"] = test["age"].fillna(mean_age)

cat_cols = ["second_language", "headset_model"]
encoder = OrdinalEncoder(dtype=int)
train[cat_cols] = encoder.fit_transform(train[cat_cols])
test[cat_cols] = encoder.transform(test[cat_cols])

scaler = StandardScaler()
train["net_worth_of_tips"] = scaler.fit_transform(train[["net_worth_of_tips"]]).round(5)
test["net_worth_of_tips"] = scaler.transform(test[["net_worth_of_tips"]]).round(5)

mapping = {"A class": 0, "B class": 1}
train["agent_class"] = train["agent_class"].map(mapping)
test["agent_class"] = test["agent_class"].map(mapping)

# formater EXACTEMENT 5 decimales en texte, uniquement sur cette colonne
train["net_worth_of_tips"] = train["net_worth_of_tips"].map(lambda x: f"{x:.5f}")
test["net_worth_of_tips"] = test["net_worth_of_tips"].map(lambda x: f"{x:.5f}")

train.to_csv("processed_train.csv", index=False)
test.to_csv("processed_test.csv", index=False)
train.head()


**RAISONNEMENT.** Regle de fuite critique : moyenne/encodeur/scaler
toujours **fit sur train uniquement**, puis `.transform()` (jamais
`.fit_transform()`) sur test. `OrdinalEncoder` garantit nativement des
codes consecutifs a partir de 0. Piege des "5 decimales exactes" : arrondir
la valeur ne suffit pas, il faut formater la colonne en chaine
(`f"{x:.5f}"`) avant `to_csv`, et seulement sur la colonne demandee (un
`float_format` global reformaterait aussi `rating`, non demande).


---
## Question 4 -- Classification -- Solution


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, confusion_matrix

full = pd.concat([train, test], ignore_index=True)

tr, va_te = train_test_split(full, test_size=0.30, random_state=42, stratify=full["agent_class"])
va, te = train_test_split(va_te, test_size=0.50, random_state=42, stratify=va_te["agent_class"])

tr.to_csv("../data/train.csv", index=False)
va.to_csv("../data/val.csv", index=False)
te.drop(columns=["agent_class"]).to_csv("../data/test.csv", index=False)

X_tr, y_tr = tr.drop(columns=["agent_class"]), tr["agent_class"]
X_va, y_va = va.drop(columns=["agent_class"]), va["agent_class"]
X_te = te.drop(columns=["agent_class"])

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_tr, y_tr)

proba_va = model.predict_proba(X_va)[:, 1]
best_threshold, best_recall = 0.5, -1
for thr in np.arange(0.1, 0.9, 0.02):
    pred = (proba_va >= thr).astype(int)
    prec = precision_score(y_va, pred, zero_division=0)
    rec = recall_score(y_va, pred, zero_division=0)
    if prec >= 0.6 and rec > best_recall:
        best_threshold, best_recall = thr, rec

pred_va = (proba_va >= best_threshold).astype(int)
print("seuil retenu:", best_threshold)
print("precision (val):", precision_score(y_va, pred_va))
print("recall (val):", recall_score(y_va, pred_va))
print(confusion_matrix(y_va, pred_va))

proba_te = model.predict_proba(X_te)[:, 1]
pred_te = (proba_te >= best_threshold).astype(int)
pd.DataFrame({"agent_class": pred_te}).to_csv("predictions.csv", index=False)


**RAISONNEMENT.** "Maximiser recall en gardant precision elevee" =
objectif asymetrique, pas un F1 standard. Deux leviers : `class_weight=
"balanced"` au fit, puis recherche de seuil sur `predict_proba` qui
maximise le recall sous contrainte de precision minimale. Evaluer sur
`val.csv` (labels connus), jamais sur `test.csv` (labels absents, comme
dans le vrai test).
